# 1. Подготовка окружения

In [1443]:
from pathlib import Path
import numpy as np
from numpy.typing import (
    ArrayLike,
    NDArray,
)

import plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = 'notebook_connected'

In [1444]:
def save_plotly_html(figure: go.Figure, path: Path, filename: str):
    figure.write_html(
        str(path),
        config={
            'toImageButtonOptions': {
                'format': 'svg',
                'filename': filename,
                'scale': 1,
            }
        },
    )

DATA_DIR = Path.cwd().parent / "data"
FIGURES_DIR = DATA_DIR / "figures"

# 2. Аналитический расчет

In [1445]:
# a_0 in Pa
amplitude: float = 0.1e6
# f_0 in Hz
frequency: float = 2.0e6
# period in s
T = 0.5e-6

# time in seconds
t_start = 0
t_end = 5e-6
dt = 1 / (50 * frequency)

TIME = np.arange(t_start, t_end, dt)

# frequency in Hz
f_start = -4.0 * frequency
f_end = +4.0 * frequency
df = 1 / (50 * T)

FREQUENCY = np.arange(f_start, f_end, df)

Исходный сигнал имеет вид:
$$
\begin{equation}
    p(t) = 2 a_0 sin (\omega_0 t) + a_0 sin (2 \omega_0 t), \quad \omega_0 = 2 \pi f_0, \quad a_0 = 0.1МПа, \quad f_0 = 2МГц
\end{equation}
$$

In [1446]:
def signal(time: ArrayLike) -> ArrayLike:
    return (2 * amplitude * np.sin(2 * np.pi * frequency * time) + amplitude * np.sin(2 * 2 * np.pi * frequency * time))

SIGNAL = signal(TIME)

Произведем аналитический расчет коэффициентов ряда Фурье:
$$
\begin{equation}
    \overline{p_{T}(f_n)} = \frac{1}{T} \int_{-\frac{T}{2}}^{\frac{T}{2}} p(t) e^{-2 \pi i f_n t} \, dt, \quad f_n = \frac{n}{T}
\end{equation}
$$

В результате получаем следующее выражение:
$$
\begin{equation}
    \overline{p_{T}(f_n)} = i a_0 \left( I(n, -1) - I(n, 1) + \frac{1}{2} I(n, -2) - \frac{1}{2} I(n, 2) \right), \quad
    I(n, k) = \frac{1}{T} \int_{-\frac{T}{2}}^{\frac{T}{2}} e^{i \omega_0 (k - n) t} \, dt = 
    \begin{cases}
        1, & n = k \\
        0, & n \neq k
    \end{cases}
\end{equation}
$$

In [1447]:
def module(f_n: ArrayLike) -> ArrayLike:
    n = f_n / frequency
    I = lambda n, k: np.where(n == k, 1, 0)
    
    return amplitude * (I(n, -1) - I(n, 1) + 0.5 * I(n, -2) - 0.5 * I(n, 2))

AMPLITUDE = module(FREQUENCY)

def phase(f_n: ArrayLike) -> ArrayLike:
    return np.pi / 2.0 + f_n * 0

PHASE = phase(FREQUENCY)

threshold = 1e-2
PHASE = np.degrees(np.unwrap(np.angle(np.fft.fftshift(PHASE))))
PHASE = np.where(np.abs(AMPLITUDE) > threshold, PHASE, 0.0)

In [1448]:
def image_part(f_n: ArrayLike) -> ArrayLike:
    n = f_n / frequency
    I = lambda n, k: np.where(n == k, 1, 0)
    
    return amplitude * (I(n, -1) - I(n, 1) + 0.5 * I(n, -2) - 0.5 * I(n, 2))

IMAGE_PART = image_part(FREQUENCY)

def real_part(f_n: ArrayLike) -> ArrayLike:
    return f_n * 0

REAL_PART = real_part(FREQUENCY)

In [1449]:
fig = make_subplots(
    rows=3, 
    cols=2,
    subplot_titles=(
        "Исходный сигнал p(t)",
        "Амплитуда",
        "Фаза",
        "Мнимая часть",
        "Действительная часть",
    ),
    specs=[
        [{"colspan": 2}, None],
        [{}, {}],              
        [{}, {}]               
    ]
)

fig.add_trace(go.Scatter(x=TIME * 1.0e6, y=SIGNAL / 1.0e6, mode="lines", marker=dict(size=8)), row=1, col=1)
fig.update_xaxes(title_text=r"Время, мкс", row=1, col=1)
fig.update_yaxes(title_text=r"Давление, МПа", row=1, col=1)

fig.add_trace(go.Scatter(x=FREQUENCY / 1.0e6, y=AMPLITUDE / 1.0e6, mode="lines", marker=dict(size=8)), row=2, col=1)
fig.update_xaxes(title_text=r"Частота, МГц", row=2, col=1)
fig.update_yaxes(title_text=r"Давление, МПа", row=2, col=1)

fig.add_trace(go.Scatter(x=FREQUENCY / 1.0e6, y=np.degrees(PHASE), mode="lines", marker=dict(size=8)), row=2, col=2)
fig.update_xaxes(title_text=r"Частота, МГц", row=2, col=2)
fig.update_yaxes(title_text=r"Угол, град.", range=[-200, +200], row=2, col=2)
fig.add_hline(y=-180, line_dash="dash", line_color="red", row=2, col=2)
fig.add_hline(y=+180, line_dash="dash", line_color="red", row=2, col=2)

fig.add_trace(go.Scatter(x=FREQUENCY / 1.0e6, y=IMAGE_PART / 1.0e6, mode="lines", marker=dict(size=8)), row=3, col=1)
fig.update_xaxes(title_text=r"Частота, МГц", row=3, col=1)
fig.update_yaxes(title_text=r"Давление, МПа", row=3, col=1)

fig.add_trace(go.Scatter(x=FREQUENCY / 1.0e6, y=REAL_PART / 1.0e6, mode="lines", marker=dict(size=8)), row=3, col=2)
fig.update_xaxes(title_text=r"Частота, МГц", row=3, col=2)
fig.update_yaxes(title_text=r"Давление, МПа", row=3, col=2)

fig.update_layout(
    title_text="Аналитический расчет",
    height=1200,
    showlegend=False,
    template="plotly_dark", 
)

save_plotly_html(
    figure=fig,
    path=FIGURES_DIR / "analytical_calculation.html",
    filename="analytical_calculation",
)

fig.show()

# 3. Выбор параметров

Найдем минимальный шаг и количество точек для временной сетки, чтобы избежать эффекта наложения частот (алиасинга). Минимальный шаг временной сетки согласно критерию Найквиста равен $dt = \frac{1}{2 f_{max}}$. В данной задаче максимальная частота в бигармоническом сигнале равна $f_{max} = 4\text{МГц}$. Таким образом, условие для оптимального шага по времени выглядит следующим образом:
$$
\begin{equation}
    dt < \frac{1}{2 f_{max}} = \frac{1}{8\text{МГц}}
\end{equation}
$$
Для получения более точных значений и красивых графиков было выбрано такое значение шага временной сетки:
$$
\begin{equation}
    dt = \frac{1}{25 f_{max}} = \frac{1}{100\text{МГц}} = 10\text{нс}.
\end{equation}
$$

Теперь рассчитаем оптимальное количество узлов временной сетки, используя выбранный шаг по времени:
$$
\begin{equation}
    dt = \frac{1}{25 f_{max}} = \frac{1}{50 f_0} = \frac{T}{50} = \frac{T}{N^*} \implies N^* = 50.
\end{equation}
$$
Для удобства и повышения точности численных расчетов лучше всего выбирать число точек равное степени двойки, поэтому окончательное оптимальное количество узлов временной сетки можно определить следующим образом:
$$
\begin{equation}
    2^n = N > N^* = 50 \implies N = 64, n = 6.
\end{equation}
$$

Теперь определим временную сетку, используя указанные выше значения.

In [1450]:
T = 0.5e-6

N = 4096
dt = 1.0e-8
t_start = 0.0
t_stop = t_start + (N - 1) * dt

TIME = np.linspace(t_start, t_stop, num=N)
TIME

array([0.000e+00, 1.000e-08, 2.000e-08, ..., 4.093e-05, 4.094e-05,
       4.095e-05])

Подготовим частотную сетку:

In [1451]:
f_s = 1 / dt

FREQUENCY = np.fft.fftshift(np.fft.fftfreq(N-1, d=dt))
FREQUENCY = np.append(FREQUENCY, f_s / 2)
FREQUENCY

array([-49987789.98778999, -49963369.96336997, -49938949.93894994, ...,
        49963369.96336997,  49987789.98778999,  50000000.        ])

Вычислим значения сеточной функции $p(l)$ в узлах выбранной сетки:

In [1452]:
SIGNAL = signal(TIME)
SIGNAL

array([      0.        ,   49935.63542935,   97913.34484314, ...,
       -252331.37362803, -236712.09402858, -212662.70208802])

# 4. Дискретное преобразование Фурье (прямое отображение)

Определим универсальную базисную функцию разложения в ряд Фурье на равномерной временной сетке:
$$
\begin{equation}
    W_N^{nl} = exp \left( i \frac{2 \pi n l}{N} \right)
\end{equation}
$$

In [1453]:
def W(N: int, n: ArrayLike, l: ArrayLike, sign: float = 1.0) -> NDArray[np.complex128]:
    return np.exp(sign * (2.0 * np.pi * 1.0j * np.outer(n, l)) / (N))

$$
\begin{equation}
    \overline{p_{T}(n)} = \frac{1}{N} \sum_{l=0}^{N-1} p(l) W_N^{-nl}
\end{equation}
$$

In [1454]:
def fft_direct_mapping(signal: ArrayLike, time: ArrayLike, N: int) -> NDArray[np.complex128]:
    signal_grid = np.asarray(signal, dtype=np.float64)
    time_grid = np.asarray(time, dtype=np.float64)
    
    n = np.arange(N)
    l = np.arange(len(time_grid))

    W_matrix = W(N=N, n=n, l=l, sign=-1.0)
    
    return (1.0 / N) * (W_matrix @ signal_grid)

In [1455]:
spectrum = fft_direct_mapping(signal=SIGNAL, time=TIME, N=4096)

SPECTRUM_AMPLITUDE = 2.0 * np.abs(np.fft.fftshift(spectrum))

threshold = 1e-2
SPECTRUM_PHASE = np.degrees(np.unwrap(np.angle(np.fft.fftshift(spectrum))))
SPECTRUM_PHASE = np.where(np.abs(spectrum) > threshold, SPECTRUM_PHASE, 0.0)

In [1456]:
fig = make_subplots(
    rows=2, 
    cols=2,
    subplot_titles=(
        "Исходный сигнал",
        "Амплитуда спектра",
        "Фаза спектра",
    ),
    specs=[
        [{"colspan": 2}, None],
        [{}, {}],
    ]
)

fig.add_trace(go.Scatter(x=TIME * 1.0e6, y=SIGNAL / 1.0e6, mode="lines", marker=dict(size=8)), row=1, col=1)
fig.update_xaxes(title_text=r"Время, мкс", row=1, col=1)
fig.update_yaxes(title_text=r"Давление, МПа", row=1, col=1)

fig.add_trace(go.Scatter(x=FREQUENCY / 1.0e6, y=SPECTRUM_AMPLITUDE / 1.0e6, mode="lines", marker=dict(size=8)), row=2, col=1)
fig.update_xaxes(title_text=r"Частота, МГц", row=2, col=1)
fig.update_yaxes(title_text=r"Давление, МПа", row=2, col=1)

fig.add_trace(go.Scatter(x=FREQUENCY / 1.0e6, y=SPECTRUM_PHASE, mode="lines", marker=dict(size=8)), row=2, col=2)
fig.update_xaxes(title_text=r"Частота, МГц", row=2, col=2)
fig.update_yaxes(title_text=r"Угол, град.", row=2, col=2)

fig.update_layout(
    title_text="Прямое преобразование Фурье",
    height=1200,
    showlegend=False,
    template="plotly_dark",
)

save_plotly_html(
    figure=fig,
    path=FIGURES_DIR / "fft_direct_mapping.html",
    filename="fft_direct_mapping",
)

fig.show()

# 5. Дискретное преобразование Фурье (обратное отображение)

In [1457]:
def fft_reverse_mapping(spectrum: ArrayLike, time: ArrayLike, N: int) -> NDArray[np.complex128]:
    spectrum_grid = np.asarray(spectrum, dtype=np.complex128)
    time_grid = np.asarray(time, dtype=np.float64)
    
    n = np.arange(N)
    l = np.arange(len(time_grid))
    
    W_matrix = W(N=N, n=l, l=n, sign=1.0)
    
    return W_matrix @ spectrum_grid

In [1458]:
SIGNAL_RECONSTRUCTED = np.real(fft_reverse_mapping(spectrum=spectrum, time=TIME, N=4096))

In [1459]:
fig = make_subplots(
    rows=2, 
    cols=1,
    subplot_titles=(
        "Исходный сигнал",
        "Восстановленный сигнал",
    ),
)

fig.add_trace(go.Scatter(x=TIME * 1.0e6, y=SIGNAL / 1.0e6, mode="lines", marker=dict(size=8)), row=1, col=1)
fig.update_xaxes(title_text=r"Время, мкс", row=1, col=1)
fig.update_yaxes(title_text=r"Давление, МПа", row=1, col=1)

fig.add_trace(go.Scatter(x=TIME * 1.0e6, y=SIGNAL_RECONSTRUCTED / 1.0e6, mode="lines", marker=dict(size=8)), row=2, col=1)
fig.update_xaxes(title_text=r"Время, мкс", row=2, col=1)
fig.update_yaxes(title_text=r"Давление, МПа", row=2, col=1)

fig.update_layout(
    title_text="Обратное преобразование Фурье",
    height=800,
    showlegend=False,
    template="plotly_dark",
)

save_plotly_html(
    figure=fig,
    path=FIGURES_DIR / "fft_reverse_mapping.html",
    filename="fft_reverse_mapping",
)

fig.show()

# 6. Сравнение результатов

В качестве эталонной реализации дискретного преобразования Фурье будем использовать алгоритмы из библиотеки `numpy`. Выполним сначала прямое отбражение:

In [1460]:
spectrum = np.fft.fftshift(np.fft.fft(SIGNAL, n=N))

SPECTRUM_AMPLITUDE = (2.0 / N) * np.abs(spectrum)

threshold = 1e-2
SPECTRUM_PHASE = np.degrees(np.unwrap(np.angle(spectrum)))
SPECTRUM_PHASE = np.where(np.abs(spectrum) > threshold, SPECTRUM_PHASE, 0.0)

Теперь применим к полученному спектру обратное преобразование Фурье и проверим качество восстановления исходного сигнала:

In [1461]:
SIGNAL_RECONSTRUCTED = np.real(np.fft.ifft(np.fft.ifftshift(spectrum)))

In [1462]:
fig = make_subplots(
    rows=3, 
    cols=2,
    subplot_titles=(
        "Исходный сигнал",
        "Амплитуда",
        "Фаза",
        "Восстановленный сигнал",
    ),
    specs=[
        [{"colspan": 2}, None],
        [{}, {}],
        [{"colspan": 2}, None],
    ]
)

fig.add_trace(go.Scatter(x=TIME * 1.0e6, y=SIGNAL / 1.0e6, mode="lines", marker=dict(size=8)), row=1, col=1)
fig.update_xaxes(title_text=r"Время, мкс", row=1, col=1)
fig.update_yaxes(title_text=r"Давление, МПа", row=1, col=1)

fig.add_trace(go.Scatter(x=FREQUENCY / 1.0e6, y=SPECTRUM_AMPLITUDE / 1.0e6, mode="lines", marker=dict(size=8)), row=2, col=1)
fig.update_xaxes(title_text=r"Частота, МГц", row=2, col=1)
fig.update_yaxes(title_text=r"Давление, МПа", row=2, col=1)

fig.add_trace(go.Scatter(x=FREQUENCY / 1.0e6, y=SPECTRUM_PHASE, mode="lines", marker=dict(size=8)), row=2, col=2)
fig.update_xaxes(title_text=r"Частота, МГц", row=2, col=2)
fig.update_yaxes(title_text=r"Угол, град.", row=2, col=2)

fig.add_trace(go.Scatter(x=TIME * 1.0e6, y=SIGNAL_RECONSTRUCTED / 1.0e6, mode="lines", marker=dict(size=8)), row=3, col=1)
fig.update_xaxes(title_text=r"Время, мкс", row=3, col=1)
fig.update_yaxes(title_text=r"Давление, МПа", row=3, col=1)

fig.update_layout(
    title_text="Эталонная реализация (numpy.fft)",
    height=1200,
    showlegend=False,
    template="plotly_dark",
)

save_plotly_html(
    figure=fig,
    path=FIGURES_DIR / "reference_implementation.html",
    filename="reference_implementation",
)

fig.show()